In [ ]:
#libs

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
import pandas as pd

df.head()


In [ ]:
df.shape


In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

#  Is the target imbalanced?
def check_target_imbalance(df, delivery_time):
  print("Target Distribution:")

  df[delivery_time].hist()  # Yeah you can just do this :)
  plt.show()
  print(df[delivery_time].value_counts())

check_target_imbalance(df, "Delivery_Time")
# as we see the score is more balanced than unbalanced. it might be showing a bit of skewness(very little)
                                                       # but for sake pf simplicity we would concedere it fully BALANCED


In [ ]:
# Task 1: Write your code here:

def check_missing_values(df):

  # Get missing values using pandas
  missing_values = df.isnull().sum()

  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

"""
We here checked for missing values, we found that our target value (Delivery_Time) has 106 Missing values
which is not ideal for a target coulmn and can add noise and may cause memorization so as THE QUESTION SAID
we will drop THE ROWS that contains those missing values in the next cell

"""


In [ ]:
#here all i want to see is the percentage of those data so i can enhance my decesions even more


# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)


#Delevery time is my target so i dont want anything to miss with that so i decided to drop the missing ROWS in there
#Im observing that the missing percentages of the rest is little not much, so i might deal with it below by means of filling missing data


In [ ]:
# Select relevant columns, we do not select 'model' (too many unique values, too sparse and will hurt the model performance)
df_clean = df.copy()

# Drop rows where target (Delivery_Time) or key features are missing - can't predict without them
#im droppin ROWS not COULOUMNS,, those couloumns are my Y labels which i need to train the model on
print(f"Before: {df.shape}")
df_clean = df_clean.dropna(subset= "Delivery_Time")
print(f"After dropping missing Delivery Times: {df_clean.shape}")


#first task is finished i identified where i can't lose important data and decided to drop those missing rows

In [ ]:
# Task 2: Write your code here:
df_clean.info()
#target value now has no missing values, the rest still does have a bit of nulls that i will deal with, i just wanted
#to make this step on its own as per importance

In [ ]:
# Fill the rest of missing values with mode - for discrete feature, mode is most representative


"""
#this is how i wanted to fill the mode for missing data:
df_clean['Weather'] = df_clean['Weather'].fillna(df_clean['Weather'].mode()[0])

print("Missing values remaining:", df_clean.isnull().sum())

But i decided a loop over the remaining missing coulmns is better
"""

for col in ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])


print("Missing values remaining:", df_clean.isnull().sum())



#as we can see below, no missing values are left
#moving on to the next step with doublicates

In [ ]:
# Task 3: Write your code here:



print("Checking for duplicates...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicates. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate  removed.")
else:
    print("No duplicate  found.")


#We Found 557 duplicates and removed them successfuly

In [ ]:
# Task 4: Write your code here:

#we have four objects :Weather,Traffic_Level, Time_of_Day, Vehicle_Type . I will encode them using labelencoder for the sake of time
#crunch and simplicity and if there still sometime ill come back and try to make it a One Hot Encoded

from sklearn.preprocessing import LabelEncoder
# Encode type columns
le = LabelEncoder()
#decided to loop again because it is easier
states = ["Weather","Traffic_Level", "Time_of_Day", "Vehicle_Type"]

for col in states:
    df_clean[col] = le.fit_transform(df_clean[col])


In [ ]:
df_clean.info()
#now we see no object data type, mission succcussful

In [ ]:
df_clean.drop(columns=['Order_ID'])
#i just remembered that having an id here wont make a differnce so i dropped it anyway before moving on to task 5


In [ ]:
#Apply feature scaling for all features (Use StandardScaler).... ill do this cell before i scale in the cell below it

#here i will start Prepareing Data for Modeling


from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


# Define features (X) and target (y)
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
                'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")



In [ ]:
# Task 5: Write your code here:
# here im going to scale
#fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

In [ ]:

Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

In [ ]:
# Task 6: Write your code here:

#  Is the target imbalanced?
def check_target_imbalance(df_clean, delivery_time):
  print("Target Distribution:")

  df_clean[delivery_time].hist()
  plt.show()
  print(df_clean[delivery_time].value_counts())

check_target_imbalance(df_clean, "Delivery_Time")
# yes it is balanced


In [ ]:
df_clean.info()

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np
from sklearn.metrics import mean_absolute_error


# here i prepared kfold for cross validation


kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)



# print avarage MAE
print("Average MAE:", np.mean(mae_scores))

In [ ]:
# Task 1: Write your code here:

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Odometer distribution
plt.figure(figsize=(10, 5))
plt.hist(df_clean['Delivery_Time'].dropna(), bins=50, edgecolor='black', color='green')
plt.title('Odometer Distribution')
plt.xlabel('Odometer')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: